<a href="https://colab.research.google.com/github/leovarconrenno/Estacao-de-Reabastecimento-de-Hidrogenio---SCADA-Core/blob/main/etapa-01-logica/04%20-%20Logica%20Proposicional%20Conectivos%20e%20Permissivos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 04 - Notebook: Implementação de Conectivos Lógicos e Permissivos de Partida

Neste notebook implementamos as funções de avaliação lógica proposicional completas (AND, OR, NOT, XOR, IMPLICATION, BICONDITIONAL) e construímos os blocos de permissivos de partida (*Start Permissives*) e intertravamento contínuo para os atuadores da **Estação de Reabastecimento de Hidrogênio**, com base nas equações definidas em `03 - tautologias e contradições.md`.

In [ ]:
from typing import Dict
import pandas as pd
import itertools

# Operadores Fundamentais da Lógica Proposicional
def NOT(p: bool) -> bool:
    return not p

def AND(p: bool, q: bool) -> bool:
    return p and q

def OR(p: bool, q: bool) -> bool:
    return p or q

def XOR(p: bool, q: bool) -> bool:
    return p ^ q

def IMPLIES(p: bool, q: bool) -> bool:
    return (not p) or q

def IFF(p: bool, q: bool) -> bool:
    return p == q

print("Operadores lógicos proposicionais carregados com sucesso.")

## Bloco Lógico de Trip de Emergência do Banco de Armazenamento (XV-10X, Setor 100)

Implementa a Seção A do documento de tautologias: $F_{1,X} \equiv p_{1,X} \lor t_{1,X} \lor g_{1,X} \lor e_{1,1}$, com a regra $F_{1,X} \rightarrow (\neg v_{1,X} \land s_{1,X} \land a_{1,1})$.

In [ ]:
def trip_banco_armazenamento(p1X: bool, t1X: bool, g1X: bool, e11: bool) -> Dict[str, bool]:
    # F_1,X ≡ p_1,X ∨ t_1,X ∨ g_1,X ∨ e_1,1
    falha = OR(OR(OR(p1X, t1X), g1X), e11)

    # F_1,X -> (¬v_1,X ∧ s_1,X ∧ a_1,1)
    valvula_aberta = NOT(falha)   # v_1,X só permanece ABERTA se não houver falha
    sinalizacao = falha           # s_1,X ACESA quando há falha
    alarme_geral = falha          # a_1,1 ATIVADO quando há falha

    return {
        'Falha_F1X': falha,
        'Valvula_1X_Aberta': valvula_aberta,
        'Sinalizacao_1X_Acesa': sinalizacao,
        'Alarme_Geral_Ativado': alarme_geral
    }

# Teste com diferentes cenários de campo (exemplo: tanque de baixa pressão, X=1)
cenarios_armazenamento = [
    {"cenario": "Operação Normal",              "args": (False, False, False, False)},
    {"cenario": "Sobrepressão (p1,1)",           "args": (True,  False, False, False)},
    {"cenario": "Sobretemperatura (t1,1)",       "args": (False, True,  False, False)},
    {"cenario": "Vazamento de H2 (g1,1)",        "args": (False, False, True,  False)},
    {"cenario": "Parada de Emergência (e1,1)",   "args": (False, False, False, True)},
    {"cenario": "Falha Múltipla (p1,1 e g1,1)",  "args": (True,  False, True,  False)},
]

resultados_armazenamento = []
for c in cenarios_armazenamento:
    res = trip_banco_armazenamento(*c["args"])
    resultados_armazenamento.append({
        "Cenário": c["cenario"],
        "Falha (F1,X)": res["Falha_F1X"],
        "Válvula 1,X Aberta": res["Valvula_1X_Aberta"],
        "Sinalização 1,X": res["Sinalizacao_1X_Acesa"],
        "Alarme Geral": res["Alarme_Geral_Ativado"]
    })

pd.DataFrame(resultados_armazenamento)

## Bloco Lógico de Permissivo do Dispensador (XV-301, Setor 300)

Implementa a Seção D (permissivo de abertura) e a Seção E (trip de abastecimento) do documento de tautologias:

$$P_{disp} \equiv h_{3,1} \land c_{3,1} \land bv_{3,1} \land \neg t_{3,1} \land p_{3,1} \land m_{2,1} \land \lnot g_{1,X} \land \lnot g_{3,1} \land \lnot e_{1,1}$$

$$F_3 \equiv t_{3,1} \lor g_{3,1} \lor \neg bv_{3,1} \lor e_{1,1}$$

In [ ]:
def permissivo_dispensador(h31: bool, c31: bool, bv31: bool, t31: bool, p31: bool,
                            m21: bool, g1X: bool, g31: bool, e11: bool) -> Dict[str, bool]:
    # Pdisp ≡ h3,1 ∧ c3,1 ∧ bv3,1 ∧ ¬t3,1 ∧ p3,1 ∧ m2,1 ∧ ¬g1,X ∧ ¬g3,1 ∧ ¬e1,1
    # (t3,1 é redefinido como 'temperatura excede o limite' = True quando RUIM,
    # mesma convenção usada em F3; por isso entra negado aqui, exigindo t3,1=False)
    permissivo = (h31 and
                  c31 and
                  bv31 and
                  NOT(t31) and
                  p31 and
                  m21 and
                  NOT(g1X) and
                  NOT(g31) and
                  NOT(e11))

    # F3 ≡ t3,1 ∨ g3,1 ∨ ¬bv3,1 ∨ e1,1  (trip imediato de abastecimento)
    trip = t31 or g31 or NOT(bv31) or e11

    # Regra operacional: Pdisp -> v3,1 (válvula só abre se o permissivo estiver satisfeito)
    valvula_dispensacao = permissivo

    return {
        'Permissivo_Habilitado': permissivo,
        'Trip_Abastecimento_Ativo': trip,
        'Valvula_301_Aberta': valvula_dispensacao
    }

# Teste com diferentes cenários de campo
# Argumentos: (h31, c31, bv31, t31, p31, m21, g1X, g31, e11)
# Observação: t31 = True significa 'temperatura excede o limite' (condição RUIM),
# logo t31 = False é a condição normal/desejada.
cenarios_dispensador = [
    {"cenario": "Operação Normal (todas condições OK)", "args": (True, True, True, False, True, True, False, False, False)},
    {"cenario": "Comando de Início Não Acionado",        "args": (False, True, True, False, True, True, False, False, False)},
    {"cenario": "Comunicação com Veículo Ausente",       "args": (True, False, True, False, True, True, False, False, False)},
    {"cenario": "Breakaway Desconectado",                "args": (True, True, False, False, True, True, False, False, False)},
    {"cenario": "Temperatura Excede o Limite (t3,1)",    "args": (True, True, True, True, True, True, False, False, False)},
    {"cenario": "Chiller Desligado (m2,1 = False)",      "args": (True, True, True, False, True, False, False, False, False)},
    {"cenario": "Vazamento na Zona de Armazenamento",    "args": (True, True, True, False, True, True, True, False, False)},
    {"cenario": "Vazamento na Zona do Dispensador",      "args": (True, True, True, False, True, True, False, True, False)},
    {"cenario": "Parada de Emergência Ativa",             "args": (True, True, True, False, True, True, False, False, True)},
]

resultados_dispensador = []
for c in cenarios_dispensador:
    res = permissivo_dispensador(*c["args"])
    resultados_dispensador.append({
        "Cenário": c["cenario"],
        "Permissivo (Pdisp)": res["Permissivo_Habilitado"],
        "Trip Abastecimento (F3)": res["Trip_Abastecimento_Ativo"],
        "Válvula 301 Aberta": res["Valvula_301_Aberta"]
    })

pd.DataFrame(resultados_dispensador)

## Geração Automática de Tabela-Verdade para Validação Exaustiva

In [ ]:
variaveis = ['h31', 'c31', 'bv31', 't31', 'p31', 'm21', 'g1X', 'g31', 'e11']
tabela = []

for combo in itertools.product([False, True], repeat=len(variaveis)):
    st = dict(zip(variaveis, combo))
    res = permissivo_dispensador(st['h31'], st['c31'], st['bv31'], st['t31'], st['p31'],
                                  st['m21'], st['g1X'], st['g31'], st['e11'])
    row = {**st, 'Permissivo': res['Permissivo_Habilitado'], 'Trip': res['Trip_Abastecimento_Ativo']}
    tabela.append(row)

df_tv = pd.DataFrame(tabela)
print(f"Total de combinações avaliadas: {len(df_tv)}")
print(f"Combinações seguras que liberam a válvula (Permissivo=True): {df_tv['Permissivo'].sum()}")
print(f"Combinações com Trip ativo: {df_tv['Trip'].sum()}")
print(df_tv.head(8))

## Verificação Formal: $P_{disp}$ e $F_3$ podem ser simultaneamente Verdadeiros?

Assim como nas provas de contradição do documento `03 - tautologias e contradições.md`, aqui verificamos exaustivamente (via tabela-verdade completa) se existe alguma combinação de entradas em que o permissivo libera a válvula ($P_{disp}=V$) **e**, ao mesmo tempo, a condição de trip está ativa ($F_3=V$) — o que representaria uma falha de projeto lógico (a válvula seria autorizada a abrir e imediatamente forçada a fechar).

In [ ]:
from IPython.display import display

conflito = df_tv[(df_tv['Permissivo'] == True) & (df_tv['Trip'] == True)]

print(f"Número de combinações onde Pdisp=V e F3=V simultaneamente: {len(conflito)}")
if len(conflito) > 0:
    print("\n⚠ ATENÇÃO: ainda existe sobreposição lógica entre o permissivo e o trip.")
    display(conflito[variaveis + ['Permissivo', 'Trip']].head(10))
else:
    print("Nenhuma sobreposição encontrada: Pdisp e F3 são mutuamente exclusivos (contradição segura).")
    print("Correção validada: com ¬t3,1 em Pdisp, a válvula só é permissionada quando a")